In [ ]:
# ============================================================
# Cell 1: 安装依赖 + 启动 Ollama (GPU) + 构建 RAG
# ✅ 新增: FAISS 缓存 (保存/加载磁盘) + folder 元数据
# ============================================================

import subprocess, os, sys

# --- 检测运行环境 ---
IS_KAGGLE = os.path.exists('/kaggle')
IS_COLAB = 'google.colab' in sys.modules
WORK_DIR = '/kaggle/working' if IS_KAGGLE else '/content'
print(f'📍 运行环境: {"Kaggle" if IS_KAGGLE else "Colab" if IS_COLAB else "其他"}')

# --- GPU 检测 ---
import torch
HAS_GPU = torch.cuda.is_available()
if HAS_GPU:
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'✅ GPU 已连接: {gpu_name} ({gpu_mem:.1f} GB)')
else:
    print('⚠️ 未检测到 GPU! 请在 Kaggle 右侧 Settings → Accelerator 选择 GPU')
    print('   或在 Colab: Runtime → Change runtime type → GPU')

# --- 系统依赖 ---
!apt-get update -qq && apt-get install -y -qq zstd ffmpeg > /dev/null 2>&1

# --- Python 依赖 ---
!pip install -q openai-whisper edge-tts aiohttp gradio langchain langchain-community faiss-cpu sentence-transformers

# --- 安装 Ollama ---
!curl -fsSL https://ollama.com/install.sh | sh

# --- 启动 Ollama (GPU 模式) ---
import time, threading

def start_ollama():
    env = os.environ.copy()
    env['OLLAMA_HOST'] = '0.0.0.0:11434'
    env['OLLAMA_ORIGINS'] = '*'
    env['OLLAMA_GPU_LAYERS'] = '999'
    subprocess.Popen(['ollama', 'serve'], env=env)

threading.Thread(target=start_ollama, daemon=True).start()
print('⏳ 等待 Ollama 启动...')

import requests
for i in range(30):
    try:
        r = requests.get('http://localhost:11434/api/tags', timeout=2)
        if r.status_code == 200:
            print(f'✅ Ollama 启动成功 (耗时 {i+1}s)')
            break
    except:
        time.sleep(1)
else:
    print('❌ Ollama 启动超时，请检查日志')

print('📥 下载 LLM 模型...')
!ollama pull qwen2.5:14b
print('✅ LLM 模型就绪')

!echo '---'; curl -s http://localhost:11434/api/tags | python3 -c "import sys,json; d=json.load(sys.stdin); [print(f'  模型: {m[\"name\"]}') for m in d.get('models',[])]"

# --- RAG 知识库构建 ---
import glob
from pathlib import Path

try:
    from langchain_core.documents import Document
except ImportError:
    from langchain.schema import Document
try:
    from langchain_text_splitters import RecursiveCharacterTextSplitter
except ImportError:
    from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# ✅ [新增] FAISS 缓存路径
FAISS_PATH = os.path.join(WORK_DIR, 'faiss_index')

# ✅ 自动检测知识库路径 (兼容 Colab / Kaggle)
possible_kb_paths = [
    '/kaggle/input/knowledge-base/RAG/data',
    '/kaggle/input',
    '/content/drive/MyDrive/Colab Notebooks/RAG/data/knowledge_base',
    os.path.join(WORK_DIR, 'knowledge_base'),
]

# Colab: 挂载 Drive
if IS_COLAB:
    try:
        from google.colab import drive
        if not os.path.exists('/content/drive'):
            drive.mount('/content/drive')
    except:
        pass

def load_hwu_documents(root_dir):
    """加载文档，✅ 新增 folder 元数据用于文件夹路由 RAG"""
    docs = []
    md_files = glob.glob(os.path.join(root_dir, '**/*.md'), recursive=True)
    print(f'  扫描: {root_dir} → {len(md_files)} 个 .md 文件')
    for fp in md_files:
        try:
            text = Path(fp).read_text(encoding='utf-8')
            lines = text.split('\n', 2)
            if len(lines) < 3:
                continue
            source = lines[0].replace('Source:', '').strip()
            topic = lines[1].replace('Topic:', '').strip()
            content = lines[2].strip()

            # ✅ [新增] 提取文件夹名作为 folder 元数据
            folder = fp.split('knowledge_base/')[-1].split('/')[0] \
                     if 'knowledge_base' in fp else topic

            docs.append(Document(
                page_content=content,
                metadata={
                    'source': source,
                    'topic': topic,
                    'filename': Path(fp).name,
                    'folder': folder,        # ✅ [新增] 供 Cell 2 文件夹路由使用
                }
            ))
        except Exception as e:
            print(f'  ⚠️ {fp}: {e}')
    return docs

# ✅ [新增] FAISS 缓存: 如果已有索引则直接加载，跳过重复构建
vector_db = None
embeddings = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2',
    model_kwargs={'device': 'cuda' if HAS_GPU else 'cpu'}
)

if os.path.exists(FAISS_PATH):
    print('📂 加载已有 FAISS 向量库...')
    vector_db = FAISS.load_local(
        FAISS_PATH, embeddings,
        allow_dangerous_deserialization=True
    )
    print('✅ 向量库从缓存加载完成')
else:
    for kb_path in possible_kb_paths:
        if os.path.exists(kb_path):
            documents = load_hwu_documents(kb_path)
            if documents:
                chunks = RecursiveCharacterTextSplitter(
                    chunk_size=1000, chunk_overlap=200
                ).split_documents(documents)
                print(f'  切分为 {len(chunks)} 个片段，构建向量索引...')
                vector_db = FAISS.from_documents(chunks, embeddings)
                # ✅ [新增] 保存到磁盘，下次直接加载
                vector_db.save_local(FAISS_PATH)
                print(f'  ✅ RAG 向量库构建完成 ({len(chunks)} chunks)，已保存到 {FAISS_PATH}')
                break

if vector_db is None:
    print('⚠️ 未找到知识库，将以纯 LLM 模式运行')
    print(f'  Kaggle: 请上传 .md 文件到 Dataset')
    print(f'  Colab: 请确认 Google Drive 中有知识库')

print('\n🎉 所有准备工作完成！请运行下一个 Cell 启动语音助手。')

In [ ]:
import subprocess, os, time, requests

os.environ['OLLAMA_HOST'] = '0.0.0.0:11434'
os.environ['OLLAMA_ORIGINS'] = '*'
subprocess.Popen(['ollama', 'serve'])

for i in range(30):
    try:
        r = requests.get('http://localhost:11434/api/tags', timeout=2)
        if r.status_code == 200:
            print(f'✅ Ollama 启动成功 ({i+1}s)')
            break
    except:
        time.sleep(1)
else:
    print('❌ 启动失败，查看日志:')
    !cat /tmp/ollama.log 2>/dev/null || echo "无日志"

In [ ]:
# ============================================================
# Cell 2: 实时语音通话 (优化版: 低延迟 + TTS 修复 + GPU 加速)
# ✅ 新增: LLM预热 | 文件夹路由RAG | 流式输出 | TTS异步分离
# ✅ 合并: 多句并行TTS | Link Protocol | 心理健康关键词扩展
# ============================================================

import gradio as gr
import json
import requests
import whisper
import edge_tts
import asyncio
import os
import time
import re
import tempfile
import shutil
import subprocess
import numpy as np
import scipy.io.wavfile as wavfile
import torch
import threading
import traceback
from concurrent.futures import ThreadPoolExecutor

# ✅ 关键修复: 允许嵌套事件循环 (Kaggle/Jupyter 自带 loop)
import nest_asyncio
nest_asyncio.apply()

# ============================================================
# Whisper 加载 (GPU)
# ============================================================
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'🖥️ Whisper 设备: {device}')

WHISPER_MODEL_SIZE = 'base'
stt_model = whisper.load_model(WHISPER_MODEL_SIZE, device=device)

if device == 'cuda':
    print(f'✅ Whisper {WHISPER_MODEL_SIZE} 已加载 (GPU)')
else:
    print(f'✅ Whisper {WHISPER_MODEL_SIZE} 已加载 (CPU - 会较慢)')

# ============================================================
# ✅ [新增 #8] LLM 后台预热
# 启动时立刻在后台发一个 hi 请求，提前把模型加载到 GPU
# ============================================================
_llm_ready = threading.Event()

def _warmup():
    print('🔥 [后台] 预热 LLM...')
    for attempt in range(3):
        try:
            r = requests.post('http://localhost:11434/api/chat', json={
                'model': 'qwen2.5:14b',
                'messages': [{'role':'user','content':'hi'}],
                'stream': False,
                'keep_alive': '60m',
                'options': {'num_predict': 3, 'num_ctx': 256}
            }, timeout=30)
            if r.status_code == 200:
                _llm_ready.set()
                print('✅ [后台] LLM 预热完成')
                return
        except Exception as e:
            print(f'⚠️ 预热 {attempt+1}: {e}')
            time.sleep(5)
    print('❌ LLM 预热失败')

threading.Thread(target=_warmup, daemon=True).start()

# ============================================================
# VAD 参数
# ============================================================
SILENCE_THRESHOLD = 0.012
SILENCE_DURATION  = 1.2
MIN_SPEECH_DURATION = 0.3

def detect_speech(audio_chunk, sr):
    if audio_chunk is None or len(audio_chunk) == 0:
        return False
    if audio_chunk.dtype == np.int16:
        audio_float = audio_chunk.astype(np.float32) / 32768.0
    elif audio_chunk.dtype == np.int32:
        audio_float = audio_chunk.astype(np.float32) / 2147483648.0
    else:
        audio_float = audio_chunk.astype(np.float32)
    rms = np.sqrt(np.mean(audio_float ** 2))
    return rms > SILENCE_THRESHOLD

def save_buffer_to_wav(audio_buffer, sr):
    if not audio_buffer:
        return None
    combined = np.concatenate(audio_buffer)
    if len(combined) < sr * MIN_SPEECH_DURATION:
        return None
    path = os.path.join(tempfile.gettempdir(), f'voice_{int(time.time()*1000)}.wav')
    if combined.dtype != np.int16:
        if np.max(np.abs(combined)) <= 1.0:
            combined = (combined * 32767).astype(np.int16)
        else:
            combined = combined.astype(np.int16)
    wavfile.write(path, sr, combined)
    return path

# ============================================================
# ASR
# ============================================================
def speech_to_text(audio_path):
    if not audio_path:
        return ''
    try:
        t0 = time.time()
        result = stt_model.transcribe(
            audio_path,
            language='en',
            fp16=(device == 'cuda'),
            condition_on_previous_text=False,
            no_speech_threshold=0.6,
            compression_ratio_threshold=2.4,
        )
        text = result['text'].strip()
        elapsed = time.time() - t0

        hallucinations = {
            'thank you', 'thanks for watching', 'subscribe',
            'you', 'bye', 'the end', '字幕', '感谢观看',
            '请不吝点赞', '订阅', '谢谢大家', ''
        }
        if text.lower().strip('.!? ') in hallucinations:
            return ''

        print(f'🎤 识别 ({elapsed:.2f}s): {text}')
        return text
    except Exception as e:
        print(f'❌ ASR Error: {e}')
        return ''

# ============================================================
# TTS (合并改进: WAV输出 + 重试3次 + 文本清理抽取)
# ============================================================
def _run_tts_sync(text, voice, output_path, retries=3):
    """将文本转为 WAV 音频文件 (✅ 重试次数 2→3)"""
    for attempt in range(retries):
        try:
            mp3_path = output_path.replace('.wav', '.mp3')
            loop = asyncio.get_event_loop()
            communicate = edge_tts.Communicate(text, voice, rate='+5%')
            loop.run_until_complete(communicate.save(mp3_path))

            if os.path.exists(mp3_path) and os.path.getsize(mp3_path) > 500:
                subprocess.run(
                    ['ffmpeg', '-y', '-loglevel', 'error',
                     '-i', mp3_path, '-ar', '22050', '-ac', '1', output_path],
                    timeout=15
                )
                try: os.remove(mp3_path)
                except: pass
                if os.path.exists(output_path) and os.path.getsize(output_path) > 500:
                    return True
            else:
                print(f'  ⚠️ TTS 文件太小 (attempt {attempt+1})')
        except Exception as e:
            print(f'  ⚠️ TTS attempt {attempt+1}: {e}')
            time.sleep(0.3)
    return False

def _clean_text_for_tts(text):
    """清理文本使其适合 TTS 朗读"""
    clean = text.split('---')[0]
    for pattern, repl in [
        (r'\(https?://[^\)]+\)', ''),
        (r'https?://\S+', ''),
        (r'For more information,?\s*visit:?\s*', ''),
        (r'[*#_>~`\[\](){}|]', ''),
        (r'\\n|\\r', ' '),
        (r'\n+', '. '),
        (r'\s{2,}', ' '),
        (r'\.{2,}', '.'),
    ]:
        clean = re.sub(pattern, repl, clean)
    return clean.strip()

def _get_tts_voice(text):
    """根据文本语言选择 TTS 语音"""
    has_cjk = bool(re.search(r'[\u4e00-\u9fff]', text))
    return 'zh-CN-XiaoxiaoNeural' if has_cjk else 'en-US-EmmaNeural'

def text_to_speech(text, voice=None):
    """完整文本 → WAV 音频。voice=None 则自动检测语言。"""
    if not text or len(text.strip()) < 2:
        return None

    clean = _clean_text_for_tts(text)
    if not clean or len(clean) < 2:
        return None

    if len(clean) > 1500:
        cut = clean[:1500].rfind('.')
        clean = clean[:cut+1] if cut > 100 else clean[:1500]

    v = voice or _get_tts_voice(clean)
    path = os.path.join(tempfile.gettempdir(), f'tts_{int(time.time()*1000)}.wav')

    if _run_tts_sync(clean, v, path):
        print(f'🔊 TTS OK: {len(clean)} chars')
        return path
    else:
        print('❌ TTS 最终失败')
        return None

# ============================================================
# ✅ 强制 Gradio 每次都重新播放音频
# ============================================================
def make_unique_audio(audio_path):
    if audio_path is None:
        return None
    try:
        ext = os.path.splitext(audio_path)[1]
        unique_path = os.path.join(tempfile.gettempdir(), f'play_{int(time.time()*1000)}{ext}')
        shutil.copy2(audio_path, unique_path)
        return unique_path
    except Exception as e:
        print(f'⚠️ copy error: {e}')
        return audio_path

# ============================================================
# ✅ 音频拼接工具 (从改进版合并)
# ============================================================
def concat_wav_files(wav_paths):
    """用 ffmpeg 拼接多个 WAV 文件为一个完整音频。
    跳过 None 和不存在的文件。返回拼接后的文件路径或 None。"""
    valid = [p for p in wav_paths if p and os.path.exists(p) and os.path.getsize(p) > 500]
    if not valid:
        return None
    if len(valid) == 1:
        return valid[0]
    
    list_path = os.path.join(tempfile.gettempdir(), f'concat_{int(time.time()*1000)}.txt')
    out_path  = os.path.join(tempfile.gettempdir(), f'joined_{int(time.time()*1000)}.wav')
    
    try:
        with open(list_path, 'w') as f:
            for p in valid:
                f.write(f"file '{p}'\n")
        
        result = subprocess.run(
            ['ffmpeg', '-y', '-loglevel', 'error',
             '-f', 'concat', '-safe', '0', '-i', list_path,
             '-ar', '22050', '-ac', '1', out_path],
            timeout=15, capture_output=True
        )
        
        try: os.remove(list_path)
        except: pass
        
        if result.returncode == 0 and os.path.exists(out_path) and os.path.getsize(out_path) > 500:
            print(f'🔗 拼接 {len(valid)} 段音频 → {out_path}')
            return out_path
        else:
            print(f'⚠️ 拼接失败: {result.stderr.decode()[:100]}')
            return valid[0]
    except Exception as e:
        print(f'⚠️ 拼接异常: {e}')
        return valid[0]

# ============================================================
# ✅ 多句并行 TTS 工具函数 (从改进版合并)
# ============================================================
_MIN_SENTENCE_LEN = 30

def tts_single_sentence(text, index, voice='en-US-EmmaNeural'):
    """对单个句子生成 TTS 音频。返回 (index, audio_path)。"""
    if not text or len(text.strip()) < 2:
        return (index, None)
    clean = _clean_text_for_tts(text)
    if not clean or len(clean) < 2:
        return (index, None)
    v = _get_tts_voice(clean) if voice is None else voice
    path = os.path.join(tempfile.gettempdir(), f'tts_chunk_{index}_{int(time.time()*1000)}.wav')
    if _run_tts_sync(clean, v, path):
        return (index, path)
    return (index, None)

# ============================================================
# ✅ [新增 #9] 文件夹路由 RAG (合并: 心理健康关键词扩展)
# ============================================================
FOLDER_ROUTER = {
    'accommodation': ['accommodation'],
    'hall': ['accommodation'],
    'housing': ['accommodation'],
    'room': ['accommodation'],
    'rent': ['accommodation'],
    'hostel': ['accommodation'],
    'fee': ['finance'],
    'tuition': ['finance'],
    'scholarship': ['finance'],
    'bursary': ['finance'],
    'funding': ['finance'],
    'loan': ['finance'],
    'money': ['finance'],
    'pay': ['finance'],
    'council tax': ['finance'],
    'enrol': ['enrollment'],
    'enroll': ['enrollment'],
    'register': ['enrollment'],
    'registration': ['enrollment'],
    'id card': ['enrollment'],
    'student card': ['enrollment'],
    'induction': ['enrollment'],
    'course': ['courses'],
    'module': ['courses'],
    'programme': ['courses'],
    'timetable': ['courses'],
    'lecture': ['courses'],
    'class': ['courses'],
    'major': ['courses'],
    'degree': ['courses'],
    'study': ['courses'],
    'exam': ['exams_assessment'],
    'assessment': ['exams_assessment'],
    'grade': ['exams_assessment'],
    'extension': ['exams_assessment'],
    'resit': ['exams_assessment'],
    'mental health': ['support'],
    'wellbeing': ['support'],
    'counsell': ['support'],
    'disability': ['support'],
    'welfare': ['support'],
    'anxiety': ['support'],
    'stress': ['support'],
    # ✅ 合并: 心理健康关键词扩展
    'depress': ['support'],
    'lonely': ['support'],
    'homesick': ['support'],
    'unhappy': ['support'],
    'sad': ['support'],
    'overwhelm': ['support'],
    'struggle': ['support'],
    'crisis': ['support'],
    'sport': ['facilities'],
    'gym': ['facilities'],
    'oriam': ['facilities'],
    'library': ['facilities'],
    'canteen': ['facilities'],
    'food': ['facilities'],
    'cafe': ['facilities'],
    'parking': ['facilities'],
    'visa': ['support', 'enrollment'],
    'international': ['enrollment', 'support'],
    'career': ['support'],
    'job': ['support'],
    'club': ['support'],
    'society': ['support'],
    'union': ['support'],
}

def get_target_folders(query):
    """根据问题关键词快速定位相关文件夹"""
    q = query.lower()
    folders = set()
    for kw, flds in FOLDER_ROUTER.items():
        if kw in q:
            folders.update(flds)
    return list(folders) if folders else None  # None 表示全库搜索

def user_wants_link(query):
    """检测用户是否在询问链接/网址"""
    q = query.lower()
    link_keywords = ['link', 'url', 'website', 'web site', 'webpage', 'web page',
                     'address', 'where can i find', 'where do i find',
                     'send me the link', 'give me the link', 'share the link',
                     'how to access', 'how do i access', 'portal',
                     '链接', '网址', '网站']
    return any(kw in q for kw in link_keywords)

# 删除第376行的 enforce_real_links（孤立引用）
# 替换为：

def enforce_real_links(text, rag_links):
    """代码级硬拦截：移除LLM编造的链接，只保留RAG中真实存在的链接"""
    # 提取RAG中的真实URL集合
    real_urls = set()
    if rag_links:
        for link_entry in rag_links:
            urls_in_entry = re.findall(r'https?://[^\s\'">\])+]+', link_entry)
            real_urls.update(u.rstrip('.,;:!?)') for u in urls_in_entry)

    def is_real_url(url):
        url_clean = url.rstrip('.,;:!?)')
        for real_url in real_urls:
            # 只允许精确匹配（忽略尾部斜杠）
            if url_clean.rstrip('/') == real_url.rstrip('/'):
                return True
        return False

    # 1) 删除Markdown链接格式 [text](url) — 保留text，处理url
    def replace_md_link(match):
        label = match.group(1)
        url = match.group(2).strip().lstrip('_').rstrip('_')  # 去掉LLM加的下划线
        if url.startswith('http') and is_real_url(url):
            return f'{label}: {url}'
        return label  # 只保留文字，删掉假链接

    text = re.sub(r'\[([^\]]*)\]\(([^)]*)\)', replace_md_link, text)

    # 2) 删除残缺的Markdown链接 [text]( 或 [text](  （没有闭合的）
    text = re.sub(r'\[([^\]]*)\]\(\s*\)', r'\1', text)  # [text]() -> text
    text = re.sub(r'\[([^\]]*)\]\($', r'\1', text, flags=re.MULTILINE)  # [text]( at line end

    # 3) 处理裸URL（可能带下划线包裹如 __https://...__）
    def replace_bare_url(match):
        url = match.group(0)
        # 去掉下划线包裹
        url_clean = url.strip('_').rstrip('.,;:!?)')
        if url_clean.startswith('http') and is_real_url(url_clean):
            return url_clean
        return ''

    text = re.sub(r'_*https?://\S+_*', replace_bare_url, text)

    # 4) 清理残留的空括号、多余空行等
    text = re.sub(r'\(\s*\)', '', text)
    text = re.sub(r'\*\s*\*', '', text)  # 空的加粗标记
    text = re.sub(r'\n\s*\n\s*\n', '\n\n', text)

    # 5) 如果没有任何真实链接，在末尾追加RAG链接
    if real_urls and not any(u in text for u in real_urls):
        top_links = list(real_urls)[:3]
        text += '\n\nRelevant links:\n' + '\n'.join(top_links)

    return text.strip()

def _diversify_by_source(docs, max_docs=6):
    """从候选文档中去重选取，确保来源URL多样性。
    每个source URL最多保留2个chunk。"""
    seen_sources = {}
    result = []
    for d in docs:
        src = d.metadata.get('source', '')
        count = seen_sources.get(src, 0)
        if count < 2:  # 每个来源最多2个chunk
            result.append(d)
            seen_sources[src] = count + 1
        if len(result) >= max_docs:
            break
    return result

def retrieve_context(query):
    """返回 (context_text, source_links)"""
    if 'vector_db' not in globals() or vector_db is None:
        return '', []
    try:
        t0 = time.time()
        target_folders = get_target_folders(query)

        if target_folders:
            candidates = vector_db.as_retriever(
                search_kwargs={'k': 20}  # 拉更多候选
            ).invoke(query)
            folder_filtered = [d for d in candidates
                    if d.metadata.get('folder','') in target_folders]
            # ✅ 来源去重：优先从不同文件取结果
            docs = _diversify_by_source(folder_filtered, max_docs=6)
            if not docs:
                docs = _diversify_by_source(candidates, max_docs=6)
        else:
            candidates = vector_db.as_retriever(
                search_kwargs={'k': 12}
            ).invoke(query)
            docs = _diversify_by_source(candidates, max_docs=6)

        parts = []
        seen_links = set()
        source_links_raw = []
        for i, d in enumerate(docs):
            source_url = d.metadata.get('source', 'N/A')
            topic = d.metadata.get('topic', 'N/A')
            # 收集去重的有效链接
            if source_url and source_url != 'N/A' and source_url.startswith('http') and source_url not in seen_links:
                seen_links.add(source_url)
                # 计算链接与查询的相关性分数：URL路径中包含查询关键词的优先
                q_words = set(query.lower().split())
                url_lower = source_url.lower()
                score = sum(1 for w in q_words if w in url_lower and len(w) > 2)
                # 路径越短（越接近主页面）得分越高
                path_depth = source_url.count('/') - 2  # 减去 https:// 的两个斜杠
                score -= path_depth * 0.1
                # 从 URL 路径提取更有意义的标题
                url_path = source_url.rstrip('/').split('/')[-1].replace('-', ' ').title()
                source_links_raw.append((score, f'{url_path}: {source_url}'))
            parts.append(
                f'--- Document {i} ---\n'
                f'Topic: {topic}\n'
                f'Source URL: {source_url}\n' # ✅ 把链接加进 Prompt
                f'Content: {d.page_content}'
            )
        # 按相关性排序，最相关的在前
        source_links_raw.sort(key=lambda x: x[0], reverse=True)
        source_links = [item[1] for item in source_links_raw]
        elapsed = time.time() - t0
        folders_str = str(target_folders) if target_folders else 'all'
        print(f'🔍 RAG ({elapsed:.2f}s) folders={folders_str} → {len(docs)} docs, {len(source_links)} links')
        return '\n\n'.join(parts), source_links
    except Exception as e:
        print(f'RAG Error: {e}')
        return '', []

# ============================================================
# ✅ LLM 调用 — generator 流式输出
# ✅ 合并: System Prompt Link Protocol (禁止编造URL)
# ============================================================
def call_llm(user_text, chat_history):
    # ✅ [新增 #8] 等待预热完成
    if not _llm_ready.is_set():
        _llm_ready.wait(timeout=120)
        if not _llm_ready.is_set():
            yield 'Still loading, please wait a moment.'
            return

    # ✅ 修复: 避免多轮上下文严重污染当前检索
    # 只有当用户输入极短（追问），且没有明确开启新话题时，才拼接上一轮问题
    recent = (chat_history or [])[-1:]
    is_short_follow_up = len(user_text.split()) <= 6 or 'link' in user_text.lower().strip()
    
    user_text_lower = user_text.lower()
    # ✅ 只要用户输入包含任何FOLDER_ROUTER中的关键词，就视为新话题，不拼接历史
    has_new_topic = any(kw in user_text_lower for kw in FOLDER_ROUTER.keys())
    if recent and recent[0][0] and is_short_follow_up and not has_new_topic:
        augmented_query = recent[0][0] + ' ' + user_text
    else:
        augmented_query = user_text
        
    ctx, _rag_links = retrieve_context(augmented_query)
    prompt = f"""
    You are the AI Student Services Assistant for Heriot-Watt University (HWU).
    You must be highly empathetic, professional, and supportive.

    Here is the retrieved knowledge from the university database:
    [CONTEXT START]
    {ctx}
    [CONTEXT END]

    [VERIFIED LINKS - You may ONLY use these URLs in your response, do NOT invent any other URLs]
    {chr(10).join(_rag_links) if _rag_links else 'No verified links available. Do NOT include any URLs in your response.'}
    [END VERIFIED LINKS]

    You must follow these prioritized guidelines:

    **CRITICAL OUTPUT RULES (YOU MUST FOLLOW STRICTLY):**
    1. "SOURCE" TAG ONLY: You are STRICTLY FORBIDDEN from inventing or recalling URLs from your memory. You MUST ONLY use the URL explicitly written after the word "Source:" at the very top of the provided [CONTEXT] documents.
    2. NO SUB-FACILITY LINKS: Do NOT provide individual links for specific facilities (like Oriam, Student Union, Health Service, etc.) unless there is a specific "Source: https://..." line for them in the context. 
    3. ONE LINK AT THE END: Provide the ONE main 'Source URL' matching the user's query at the very bottom of your response in plain text (e.g., "Main Source: https://...").
    4. RAW URLS ONLY: NEVER use Markdown formatting (e.g., [Text](URL)). Write the naked URL out in plain text.
    
    **Priority 1: Crisis Intervention & Mental Health**
    If the user expresses distress, depression, anxiety, or a crisis:
    - Respond with deep empathy and humanistic care.
    - Validate their feelings and prioritize their safety.
    - IMMEDIATELY suggest they contact HWU Student Wellbeing Services.
    - Do NOT say "I apologize, I can only answer...". Be a supportive human.

    **Priority 2: Daily Conversation & Greetings**
    If the user says hello, asks how you are, or engages in basic small talk:
    - Respond warmly and naturally.
    - Gently guide the conversation back to how you can help them with Heriot-Watt University student services.

    **Priority 3: HWU Student Services Queries**
    If the question is about HWU (accommodation, enrollment, campus life, etc.):
    - Provide a RICH, detailed, and structured answer using the [CONTEXT].
    - Expand on the details to make the answer comprehensive rather than overly brief.
    - If any LINK in the [CONTEXT] is relevant, naturally weave it into your answer.
    - Pay close attention to ALL documents retrieved — important details like URLs, dates, and contact info may appear in any of them.

    **Priority 4: Out of Scope Queries**
    If the question is completely unrelated to HWU or student services (e.g., coding, math, global news):
    - Politely decline by saying: "I'd love to chat about that, but my expertise is focused on Heriot-Watt University Student Services. How can I help you with your university life today?"

    Output Guidelines:
    - ALWAYS reply in the same language the user is speaking (e.g., if asked in Chinese, reply in natural Chinese).
    - Maintain a conversational and rich tone.
    """

    msgs = [{'role': 'system', 'content': prompt}]
    recent_history = (chat_history or [])[-3:]
    for u, a in recent_history:
        msgs.append({'role': 'user', 'content': u})
        if a:
            # ✅ 清洗历史回复中的URL，防止LLM从历史中"回忆"出旧链接
            clean_a = re.sub(r'https?://\S+', '', a)
            clean_a = re.sub(r'Main Source:.*', '', clean_a).strip()
            msgs.append({'role': 'assistant', 'content': clean_a})
    msgs.append({'role': 'user', 'content': user_text})

    try:
        t0 = time.time()
        resp = requests.post(
            'http://localhost:11434/api/chat',
            json={
                'model': 'qwen2.5:14b',
                'messages': msgs,
                'stream': True,              # ✅ 流式
                'keep_alive': '60m',
                'options': {
                    'num_predict': 400,
                    'temperature': 0.6,
                    'num_ctx': 2048,
                }
            },
            timeout=120,
            stream=True
        )

        # ✅ 流式 yield: 每个 token 立即推送
        full_text = ''
        for line in resp.iter_lines():
            if line:
                try:
                    chunk = json.loads(line)
                    token = chunk.get('message',{}).get('content','')
                    if token:
                        full_text += token
                        yield full_text      # ← 每次 yield 累积文本
                    if chunk.get('done', False):
                        break
                except json.JSONDecodeError:
                    continue

        elapsed = time.time() - t0
        print(f'🤖 LLM ({elapsed:.1f}s): {full_text[:100]}...')
        
        # ✅ 使用代码级硬拦截，剔除所有幻觉链接，只保留纯正文或真实的内嵌链接
        if full_text:
            full_text = enforce_real_links(full_text, _rag_links)
            yield full_text

        if not full_text:
            yield 'Sorry, I could not generate a response.'

    except requests.Timeout:
        yield 'Sorry, the response timed out. Please try again.'
    except Exception as e:
        print(f'❌ LLM Error: {e}')
        yield f'Sorry, service error: {e}'

# ============================================================
# ✅ 多句并行 TTS 流水线 (从改进版合并)
#
# 原理:
#   LLM 流式输出: [===句1===][===句2===][===句3===]...
#   TTS 句1:       [==TTS==]                           ← 与 LLM 并行
#   TTS 句2:                  [==TTS==]                ← 与 LLM 并行
#   TTS 句3:                              [==TTS==]   ← LLM结束后可能已完成
#   拼接:                                          [concat] → 完整音频
# ============================================================
def stream_llm_with_parallel_tts(user_text, chat_history):
    """
    Stream LLM output sentence-by-sentence.
    Each completed sentence is immediately submitted to TTS in a thread pool.
    After LLM finishes, collect all TTS audio in order and concatenate.
    Returns: (full_text, audio_path)
    """
    full_text = ''
    current_buffer = ''
    sentence_index = 0
    tts_executor = ThreadPoolExecutor(max_workers=3)
    tts_futures = []

    voice = _get_tts_voice(user_text)

    for partial in call_llm(user_text, chat_history):
        new_tokens = partial[len(full_text):]
        full_text = partial
        current_buffer += new_tokens

        if (re.search(r'[.!?。！？]\s*$', current_buffer)
                and len(current_buffer.strip()) >= _MIN_SENTENCE_LEN):
            sentence = current_buffer.strip()
            future = tts_executor.submit(tts_single_sentence, sentence, sentence_index, voice)
            tts_futures.append((sentence_index, future))
            print(f'  📝 Chunk {sentence_index}: "{sentence[:60]}..." → TTS submitted')
            sentence_index += 1
            current_buffer = ''

    # Flush remaining text
    if current_buffer.strip() and len(current_buffer.strip()) >= 2:
        future = tts_executor.submit(tts_single_sentence, current_buffer.strip(), sentence_index, voice)
        tts_futures.append((sentence_index, future))
        print(f'  📝 Chunk {sentence_index} (final): "{current_buffer.strip()[:60]}..."')

    if not full_text:
        tts_executor.shutdown(wait=False)
        return 'Sorry, I could not generate a response.', None

    print(f'🤖 LLM done, {len(tts_futures)} TTS chunks, waiting for remaining TTS...')

    # Collect TTS results IN ORDER
    tts_futures.sort(key=lambda x: x[0])
    audio_paths = []
    for idx, future in tts_futures:
        try:
            _, audio_path = future.result(timeout=30)
            if audio_path:
                audio_paths.append(audio_path)
        except Exception as e:
            print(f'  ⚠️ TTS chunk {idx} failed: {e}')

    tts_executor.shutdown(wait=False)

    final_audio = concat_wav_files(audio_paths)

    # ✅ 清理临时 chunk 文件
    for p in audio_paths:
        if p and p != final_audio:
            try: os.remove(p)
            except: pass

    return full_text, final_audio

# ============================================================
# 核心: 流式音频 VAD 回调 (✅ 绝对安全版: 修复前端崩溃)
# ============================================================
def process_streaming_audio(audio_chunk, state_data):
    try:
        if state_data is None:
            state_data = {
                'audio_buffer': [], 'is_speaking': False, 'silence_start': None,
                'speech_detected': False, 'chat_history': [], 'processing': False
            }

        # 如果正在处理大段文字，直接返回 None 避免前端播放器组件报错
        if state_data.get('processing', False):
            return state_data, state_data.get('chat_history', []), None, '🔄 AI is processing...'

        if audio_chunk is None:
            s = '🎙️ Waiting for you to speak...' if _llm_ready.is_set() else '⏳ LLM loading...'
            return state_data, state_data.get('chat_history', []), None, s

        if not isinstance(audio_chunk, tuple) or len(audio_chunk) != 2:
            return state_data, state_data.get('chat_history', []), None, '⚠️ Audio format mismatch'

        sr, audio_data = audio_chunk
        
        if audio_data is None or len(audio_data) == 0:
            return state_data, state_data.get('chat_history', []), None, '⚠️ Empty audio chunk'

        if len(audio_data.shape) > 1:
            audio_data = audio_data[:, 0]

        now = time.time()
        is_voice = detect_speech(audio_data, sr)

        if is_voice:
            state_data['audio_buffer'].append(audio_data)
            state_data['is_speaking'] = True
            state_data['speech_detected'] = True
            state_data['silence_start'] = None
            return state_data, state_data.get('chat_history', []), None, '🗣️ Listening...'
        else:
            if state_data['speech_detected']:
                state_data['audio_buffer'].append(audio_data)
                if state_data['silence_start'] is None:
                    state_data['silence_start'] = now

                silence_elapsed = now - state_data['silence_start']

                if silence_elapsed >= SILENCE_DURATION:
                    state_data['processing'] = True
                    print(f'\n⏹️ 静音 {silence_elapsed:.1f}s，开始处理...')

                    wav_path = save_buffer_to_wav(state_data['audio_buffer'], sr)
                    state_data['audio_buffer'] = []
                    state_data['is_speaking'] = False
                    state_data['speech_detected'] = False
                    state_data['silence_start'] = None

                    if wav_path is None:
                        state_data['processing'] = False
                        return state_data, state_data.get('chat_history', []), None, '🎙️ Audio too short...'

                    pipeline_start = time.time()
                    user_text = speech_to_text(wav_path)
                    try: os.remove(wav_path)
                    except: pass

                    if not user_text:
                        state_data['processing'] = False
                        return state_data, state_data.get('chat_history', []), None, '🎙️ Could not hear clearly...'

                    ai_text, audio_out = stream_llm_with_parallel_tts(
                        user_text, state_data.get('chat_history', [])
                    )

                    total_latency = time.time() - pipeline_start
                    print(f'⏱️ 总延迟: {total_latency:.1f}s')

                    history = state_data.get('chat_history', [])
                    history.append((user_text, ai_text))
                    state_data['chat_history'] = history
                    state_data['processing'] = False

                    # 只有真正生成了新音频，才传给前端播放器
                    return state_data, history, make_unique_audio(audio_out), f'🎙️ Done ({total_latency:.1f}s)'
                else:
                    remaining = SILENCE_DURATION - silence_elapsed
                    return state_data, state_data.get('chat_history', []), None, f'🤫 Paused... ({remaining:.1f}s)'
            else:
                s = '🎙️ Waiting for you to speak...' if _llm_ready.is_set() else '⏳ LLM loading...'
                return state_data, state_data.get('chat_history', []), None, s
                
    except Exception as e:
        import traceback
        print(f"❌ 后台崩溃: {traceback.format_exc()}")
        if 'state_data' in locals() and state_data is not None:
            state_data['processing'] = False
        return state_data, state_data.get('chat_history', []), None, f"❌ Error: Check terminal"

# ============================================================
# ✅ 文字输入: 流式显示 + 多句并行 TTS (防崩溃版)
# ============================================================
def handle_text_input(user_text, state_data):
    if state_data is None:
        state_data = {
            'audio_buffer': [], 'is_speaking': False, 'silence_start': None,
            'speech_detected': False, 'chat_history': [], 'processing': False
        }
    if not user_text or not user_text.strip():
        yield state_data, state_data.get('chat_history', []), gr.skip(), '🎙️ Waiting for you to speak...', ''
        return

    if not _llm_ready.is_set():
        _llm_ready.wait(timeout=120)

    voice = _get_tts_voice(user_text)
    tts_executor = ThreadPoolExecutor(max_workers=3)
    tts_futures = []
    sentence_index = 0
    submitted_up_to = 0

    temp_history = state_data.get('chat_history', []) + [(user_text, '▌')]
    yield state_data, temp_history, gr.skip(), '🤖 Generating...', ''

    partial = ''
    for partial in call_llm(user_text, state_data.get('chat_history', [])):
        temp_history = state_data.get('chat_history', []) + [(user_text, partial + '▌')]
        yield state_data, temp_history, gr.skip(), '🤖 Generating...', ''

        current_buffer = partial[submitted_up_to:]
        if (re.search(r'[.!?。！？]\s*$', current_buffer)
                and len(current_buffer.strip()) >= _MIN_SENTENCE_LEN):
            sentence = current_buffer.strip()
            future = tts_executor.submit(tts_single_sentence, sentence, sentence_index, voice)
            tts_futures.append((sentence_index, future))
            print(f'  📝 [文字] Chunk {sentence_index}: "{sentence[:60]}..." → TTS')
            sentence_index += 1
            submitted_up_to = len(partial)

    remaining = partial[submitted_up_to:].strip()
    if remaining and len(remaining) >= 2:
        future = tts_executor.submit(tts_single_sentence, remaining, sentence_index, voice)
        tts_futures.append((sentence_index, future))
        print(f'  📝 [文字] Chunk {sentence_index} (final): "{remaining[:60]}..."')

    history = state_data.get('chat_history', [])
    history.append((user_text, partial))
    state_data['chat_history'] = history
    final_history = history.copy()
    yield state_data, final_history, gr.skip(), '⏳ Synthesizing voice...', ''

    tts_futures.sort(key=lambda x: x[0])
    audio_paths = []
    for idx, future in tts_futures:
        try:
            _, audio_path = future.result(timeout=30)
            if audio_path:
                audio_paths.append(audio_path)
        except Exception as e:
            print(f'  ⚠️ TTS chunk {idx} failed: {e}')

    tts_executor.shutdown(wait=False)
    final_audio = concat_wav_files(audio_paths)

    for p in audio_paths:
        if p and p != final_audio:
            try: os.remove(p)
            except: pass

    # 只有全部合成完毕，才把音频塞进播放器
    yield state_data, final_history, make_unique_audio(final_audio), '🎙️ AI replied, you can speak now...', ''


# ============================================================
# Gradio 界面 (完美对齐 + 渐变标题 + 深夜模式)
# ============================================================

custom_css = """
/* 隐藏 Gradio 顶部的默认留白和页脚水印 */
footer { display: none !important; }
.gradio-container { border: none !important; background: transparent !important; }

/* 基础背景 (白天模式) - 保留你喜欢的弥散渐变 */
body, .gradio-container {
    background-color: #fafafa !important;
    background-image: 
        radial-gradient(circle at 50% 100%, rgba(244, 114, 182, 0.25) 0%, transparent 50%),
        radial-gradient(circle at 0% 0%, rgba(167, 139, 250, 0.15) 0%, transparent 50%) !important;
    font-family: 'Inter', system-ui, -apple-system, sans-serif !important;
    transition: background-color 0.4s ease, background-image 0.4s ease !important;
}

body.dark-mode, body.dark-mode .gradio-container {
    background-color: #0f172a !important;
    background-image: 
        radial-gradient(circle at 50% 100%, rgba(190, 24, 93, 0.2) 0%, transparent 50%),
        radial-gradient(circle at 0% 0%, rgba(76, 29, 149, 0.2) 0%, transparent 50%) !important;
}

.ui-card {
    background: rgba(255, 255, 255, 0.85) !important;
    border-radius: 24px !important;
    box-shadow: 0 10px 40px -10px rgba(0, 0, 0, 0.08) !important;
    border: 1px solid rgba(255, 255, 255, 1) !important;
    padding: 20px !important;
    margin-bottom: 15px !important;
    transition: background 0.4s ease, border-color 0.4s ease !important;
}
body.dark-mode .ui-card {
    background: rgba(30, 41, 59, 0.85) !important;
    border: 1px solid rgba(255, 255, 255, 0.05) !important;
    box-shadow: 0 10px 40px -10px rgba(0, 0, 0, 0.4) !important;
}

.dyn-text { color: #475569 !important; transition: color 0.4s ease; }
body.dark-mode .dyn-text { color: #cbd5e1 !important; }
/* 核心修复：增加了 , .dyn-subtext * 通配符选择器 */
.dyn-subtext, .dyn-subtext * { color: #64748b !important; transition: color 0.4s ease; }
body.dark-mode .dyn-subtext, body.dark-mode .dyn-subtext * { color: #f8fafc !important; }

.gradient-title {
    font-size: 2.8rem !important;
    font-weight: 800 !important;
    background: linear-gradient(90deg, #c026d3, #8b5cf6) !important;
    -webkit-background-clip: text !important;
    -webkit-text-fill-color: transparent !important;
    margin-bottom: 0 !important;
}
body.dark-mode .gradient-title { background: linear-gradient(90deg, #e879f9, #a78bfa) !important; -webkit-background-clip: text !important; -webkit-text-fill-color: transparent !important; }

.subtitle {
    font-size: 1.2rem !important;
    font-weight: 600 !important;
    color: #334155 !important;
    margin-top: 5px !important;
}
body.dark-mode .subtitle { color: #e2e8f0 !important; }

.input-row {
    display: flex !important;
    align-items: center !important;
    gap: 12px !important;
    padding: 10px 15px !important;
}
.input-box { flex-grow: 1 !important; margin: 0 !important; }
.input-box > div { border: none !important; box-shadow: none !important; background: transparent !important; }
.input-box textarea {
    border-radius: 12px !important;
    border: 1px solid #cbd5e1 !important;
    padding: 12px 15px !important;
    font-size: 15px !important;
    transition: all 0.3s ease !important;
}
body.dark-mode .input-box textarea { background: #1e293b !important; border-color: #475569 !important; color: #f8fafc !important; }
.input-box textarea:focus { border-color: #8b5cf6 !important; outline: none !important; }

.send-btn {
    margin: 0 !important;
    height: 48px !important;
    border-radius: 12px !important;
    background: linear-gradient(90deg, #ec4899, #8b5cf6) !important;
    color: white !important;
    font-weight: 600 !important;
    border: none !important;
    min-width: 140px !important;
    transition: transform 0.2s ease, box-shadow 0.2s ease !important;
}
.send-btn:hover { transform: translateY(-2px); box-shadow: 0 8px 20px -5px rgba(236, 72, 153, 0.4) !important; }

.theme-btn {
    border-radius: 12px !important;
    background: transparent !important;
    border: 1px solid #cbd5e1 !important;
    color: #475569 !important;
    transition: all 0.3s ease !important;
}
body.dark-mode .theme-btn { border-color: #475569 !important; color: #cbd5e1 !important; background: rgba(30, 41, 59, 0.5) !important; }

#chatbot { background: transparent !important; border: none !important; }
#chatbot .message { border-radius: 16px !important; }
"""

theme = gr.themes.Soft(
    primary_hue="pink", 
    secondary_hue="purple",
    neutral_hue="slate"
)

with gr.Blocks(title='HWU Voice Call', css=custom_css, theme=theme) as demo:
    
    with gr.Row():
        with gr.Column(scale=9):
            gr.HTML(f"""
                <div style="text-align: center; margin-top: 2rem; margin-bottom: 2.5rem;">
                    <h1 class="gradient-title">HWU AI Student Services</h1>
                    <h2 class="subtitle">Live Voice Call</h2>
                    <p class="dyn-subtext" style="font-size: 1.05rem; max-width: 650px; margin: 1.5rem auto 0; line-height: 1.6;">
                        <strong>Instructions:</strong> Click the microphone to start the call, speak directly, and the AI will automatically reply after a {SILENCE_DURATION}s pause.<br>
                        <span style="font-size: 0.85em; opacity: 0.8;">Environment: GPU={torch.cuda.is_available()} | Whisper={WHISPER_MODEL_SIZE} | Device={device} | MultiSentenceTTS=ON</span>
                    </p>
                </div>
            """)
        with gr.Column(scale=1):
            theme_btn = gr.Button("🌓 Theme", elem_classes="theme-btn")
            theme_btn.click(None, None, None, js="""
                function() {
                    document.body.classList.toggle('dark-mode');
                }
            """)

    call_data = gr.State(value={
        'audio_buffer': [], 'is_speaking': False, 'silence_start': None,
        'speech_detected': False, 'chat_history': [], 'processing': False
    })

    with gr.Row():
        with gr.Column(scale=3, elem_classes="ui-card"):
            chatbot = gr.Chatbot(label='Conversation History', height=420, elem_id="chatbot", type='tuples')
            
        with gr.Column(scale=1, elem_classes="ui-card"):
            status = gr.Textbox(value='⏳ LLM loading...', label='Call Status', interactive=False)
            audio_player = gr.Audio(label='🔊 AI Response', autoplay=True, interactive=False, type='filepath')
            
            gr.Markdown('<hr style="margin: 15px 0; border-color: #e2e8f0;">')
            
            mic = gr.Audio(
                sources=['microphone'], streaming=True,
                label='🎤 Microphone (Click to start)'
            )
            clear_btn = gr.Button('🗑️ Clear Conversation', variant='stop')

    with gr.Row(elem_classes="ui-card input-row"):
        text_input = gr.Textbox(
            show_label=False,
            container=False,
            placeholder='Or type your questions here...',
            scale=4,
            elem_classes="input-box"
        )
        send_btn = gr.Button('Send Message', scale=1, elem_classes="send-btn")

    # --- 核心事件绑定 ---
    mic.stream(
        fn=process_streaming_audio,
        inputs=[mic, call_data],
        outputs=[call_data, chatbot, audio_player, status],
        concurrency_limit=1  # ✅ 新增：强制单线程处理音频流，防止前端洪水攻击
    )
    send_btn.click(
        fn=handle_text_input,
        inputs=[text_input, call_data],
        outputs=[call_data, chatbot, audio_player, status, text_input]
    )
    text_input.submit(
        fn=handle_text_input,
        inputs=[text_input, call_data],
        outputs=[call_data, chatbot, audio_player, status, text_input]
    )

    def clear_all():
        return (
            {'audio_buffer': [], 'is_speaking': False, 'silence_start': None,
             'speech_detected': False, 'chat_history': [], 'processing': False},
            [], None, '🎙️ Session Cleared'
        )
    clear_btn.click(fn=clear_all, outputs=[call_data, chatbot, audio_player, status])

print('\n🚀 启动实时语音通话 (多句并行TTS)...')

# 先测试 share 服务器是否可达
_use_share = False
try:
    import urllib.request
    urllib.request.urlopen('https://api.gradio.app/', timeout=10)
    _use_share = True
    print('✅ Gradio share 服务器可达')
except Exception as _e:
    print(f'⚠️ Gradio share 服务器不可达 ({_e})，使用 local 模式')

if _use_share:
    demo.launch(debug=True, share=True)
else:
    demo.launch(debug=True, share=False, server_name='0.0.0.0', server_port=7860)
    print('💡 如在 Kaggle，可在右侧面板查看 Web Preview (端口 7860)')
